# Tarea 2 · Voces y documentos peruanos: del archivo escaneado al texto buscable
## Parte 2 — Evaluación (CER/WER), eficiencia y extracción estructurada

**Autora:** Mirian de la Cruz Condori · **Notebook:** `02_evaluacion.ipynb`

> **Pregunta guía:** ¿Podemos convertir documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

Este notebook mide **qué tan bien** se extrae el texto. Compara dos OCR (**PaddleOCR**, el principal, y **Tesseract**, el alternativo) y dos ASR (**faster-whisper small**, el principal, y **Parakeet TDT 0.6B v3**, el alternativo) sobre una **muestra fija** de 10 documentos y 10 clips, contra referencias revisadas por la autora. También evalúa la eficiencia (tiempo, RTF y estabilidad) y la extracción estructurada con un LLM local.

### Metodología
1. **Muestra congelada antes de mirar resultados:** se eligió con semilla 20261024 a partir de metadatos (tipo y fuente) y se versionó en el commit `cddecce`, antes de ejecutar los modelos alternativos.
2. **Referencias:** la autora revisó las 20 referencias. El validador las acepta solo si su estado (`REVISADO_SIN_CAMBIOS` o `REVISADO_CON_CAMBIOS`) coincide con la huella del borrador. La sección 2 declara el origen de cada una.
3. **Normalización idéntica** para la referencia y para todos los modelos (`scripts/evaluacion.py`), validada con pruebas sintéticas.
4. **Métricas:**
   - $\text{CER} = \frac{S + D + I}{N_{\text{caracteres}}}$ y $\text{WER} = \frac{S + D + I}{N_{\text{palabras}}}$: sustituciones, eliminaciones e inserciones sobre la longitud de la referencia. **Menor es mejor.**
   - $\text{RTF} = \frac{\text{tiempo de procesamiento}}{\text{duración del audio}}$: un valor menor que 1 indica que procesa más rápido que el tiempo real. **Menor es mejor.**
   - Como hay solo 10 elementos por modalidad, las diferencias entre modelos se acompañan de **intervalos de confianza por bootstrap pareado**.
5. **Sin re-ejecutar modelos pesados:** todas las predicciones provienen de las cachés de la Parte 1 y de `scripts/10_predicciones_evaluacion.py`. Los tiempos provienen de la medición controlada `scripts/11_benchmark_tiempos.py`, con los cuatro modelos en la misma sesión.

## 0. Configuración

In [1]:
import sys, json, subprocess, importlib.util, warnings
from datetime import datetime
from pathlib import Path
RAIZ = Path.cwd()
sys.path.insert(0, str(RAIZ / "scripts"))
warnings.filterwarnings("ignore")
import logging
for _n in ["kaleido", "choreographer", "logistro"]:
    logging.getLogger(_n).setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, Markdown
import jiwer

import evaluacion            # normalización y fórmulas (CER/WER/RTF)
import metricas              # cálculo sobre la muestra congelada
import graficos
from utilidades import DATOS, RAW, REPORTES

crono = graficos.Cronometro(); crono.marca("0. Configuración")
PENDIENTE = "PENDIENTE_REVISION_HUMANA"
pd.set_option("display.max_colwidth", 140, "display.width", 200)
print("Ejecución:", datetime.now().strftime("%Y-%m-%d %H:%M"))

TINTA, TINTA2, TENUE, GRID, EJE, SUPERFICIE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
CATEGORICA = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
# Regla de color: modelo principal = azul (posición 1), modelo alternativo = naranja (posición 2)
COLOR_OCR = {"PaddleOCR": CATEGORICA[0], "Tesseract": CATEGORICA[1]}
COLOR_ASR = {"faster-whisper small": CATEGORICA[0], "Parakeet TDT 0.6B v3": CATEGORICA[1]}
ORDEN_TIPO = ["impreso", "foto", "manuscrito"]
FUENTE_TXT = "Segoe UI, system-ui, -apple-system, sans-serif"
pio.templates["tarea2"] = go.layout.Template(layout=dict(
    font=dict(family=FUENTE_TXT, size=13, color=TINTA2), paper_bgcolor=SUPERFICIE, plot_bgcolor=SUPERFICIE,
    colorway=CATEGORICA, title=dict(x=0.01, xanchor="left", font=dict(size=18, color=TINTA)),
    xaxis=dict(gridcolor=GRID, linecolor=EJE, zeroline=False, ticks="", tickfont=dict(color=TENUE)),
    yaxis=dict(gridcolor=GRID, linecolor=EJE, zeroline=False, ticks="", tickfont=dict(color=TINTA2)),
    hoverlabel=dict(bgcolor="white", bordercolor=EJE, font=dict(family=FUENTE_TXT, color=TINTA, size=12)),
    legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="left", x=0, font=dict(color=TINTA2)),
    margin=dict(l=20, r=30, t=110, b=50), bargap=0.3, bargroupgap=0.08))
pio.templates.default = "plotly_white+tarea2"
pio.renderers.default = "plotly_mimetype+notebook_connected"

Ejecución: 2026-10-25 21:02


## 1. Muestra congelada
Se verifica en vivo que la selección reproducida con la semilla sea **idéntica** al CSV versionado, y se muestra en qué commit quedó registrada.

In [2]:
crono.marca("1. Muestra congelada")
spec = importlib.util.spec_from_file_location("m08", RAIZ / "scripts" / "08_muestra_evaluacion.py")
m08 = importlib.util.module_from_spec(spec); spec.loader.exec_module(m08)
muestra = pd.read_csv(DATOS / "muestra_evaluacion.csv")
reproducida = m08.seleccionar()
assert sorted(reproducida["id"]) == sorted(muestra["id"]), "La muestra reproducida no coincide con la congelada"
commit = subprocess.run(["git", "log", "--format=%h %ad %s", "--date=format:%Y-%m-%d %H:%M", "-1", "--",
                         "datos/muestra_evaluacion.csv"], capture_output=True, text=True, encoding="utf-8", cwd=RAIZ).stdout.strip()
print(f"Semilla {m08.SEMILLA} · huella SHA-256 {m08.huella(muestra)[:16]}… · reproducción idéntica ✅")
print("Congelada en:", commit)
print("Excluido a priori:", muestra["excluidos_a_priori"].iloc[0])
display(muestra.groupby(["modalidad", "estrato"])["id"].agg(lambda s: ", ".join(s)).rename("ids").reset_index()
        .style.hide(axis="index"))

Semilla 20261024 · huella SHA-256 589dbb761fadd1c6… · reproducción idéntica ✅
Congelada en: cddecce 2026-10-24 10:01 eval: congela muestra fija de evaluación (10 documentos + 10 clips)
Excluido a priori: doc_025: mapa con ~1 400 rótulos: referencia manual completa no factible


modalidad,estrato,ids
audio,audio_A,"aud_002, aud_006"
audio,audio_B,aud_010
audio,audio_C,aud_012
audio,audio_D,aud_013
audio,audio_E,aud_019
audio,audio_F,"aud_022, aud_023, aud_026"
audio,audio_G,aud_028
documento,foto,"doc_008, doc_011, doc_019"
documento,impreso,"doc_005, doc_022, doc_023"
documento,manuscrito,"doc_001, doc_009, doc_015, doc_016"


## 2. Referencias: origen, revisión y trazabilidad
La validez del CER y el WER depende de las referencias, así que esta sección declara de dónde proviene cada una y cuánto la modificó la revisión humana. El texto de las referencias es privado: puede contener nombres reales, está en `datos/raw/` y no se versiona.

In [3]:
crono.marca("2. Referencias y trazabilidad")
control = pd.read_csv(REPORTES / "control_referencias.csv")
estado = pd.read_csv(DATOS / "referencias_estado.csv")
tabla_ref = control.merge(estado[["id", "cambio_vs_borrador", "consistencia", "apta_para_metricas", "n_palabras"]], on="id")
tabla_ref["origen"] = np.where(tabla_ref["modalidad"] == "documento",
                               "borrador visual LLM (Claude)", "borrador ASR auxiliar (wav2vec2, no evaluado)")
cambio = []
for _, r in tabla_ref.iterrows():
    sub = "documentos" if r["modalidad"] == "documento" else "audio"
    borrador = (RAW / "referencias" / "borradores_snapshot" / f"{r['id']}.txt").read_text(encoding="utf-8")
    final = (RAW / "referencias" / sub / f"{r['id']}.txt").read_text(encoding="utf-8")
    if borrador == final:
        cambio.append(0.0)
    else:  # magnitud de la corrección humana: distancia normalizada entre borrador y referencia final
        cambio.append(evaluacion.wer(final, borrador)["wer"] if sub == "audio" else evaluacion.cer(final, borrador)["cer"])
tabla_ref["magnitud_correccion"] = cambio
display(tabla_ref[["id", "modalidad", "origen", "estado", "cambio_vs_borrador", "magnitud_correccion", "consistencia",
                   "apta_para_metricas"]].style.hide(axis="index").format({"magnitud_correccion": "{:.3f}"}))
resumen_ref = tabla_ref.groupby(["modalidad", "estado"]).size().rename("n").reset_index()
display(resumen_ref.style.hide(axis="index"))
assert tabla_ref["apta_para_metricas"].all() and (tabla_ref["consistencia"] == "ok").all()

id,modalidad,origen,estado,cambio_vs_borrador,magnitud_correccion,consistencia,apta_para_metricas
doc_005,documento,borrador visual LLM (Claude),REVISADO_SIN_CAMBIOS,igual_al_borrador,0.000,ok,True
doc_022,documento,borrador visual LLM (Claude),REVISADO_SIN_CAMBIOS,igual_al_borrador,0.000,ok,True
doc_023,documento,borrador visual LLM (Claude),REVISADO_SIN_CAMBIOS,igual_al_borrador,0.000,ok,True
doc_008,documento,borrador visual LLM (Claude),REVISADO_SIN_CAMBIOS,igual_al_borrador,0.000,ok,True
doc_011,documento,borrador visual LLM (Claude),REVISADO_SIN_CAMBIOS,igual_al_borrador,0.000,ok,True
doc_019,documento,borrador visual LLM (Claude),REVISADO_SIN_CAMBIOS,igual_al_borrador,0.000,ok,True
doc_001,documento,borrador visual LLM (Claude),REVISADO_SIN_CAMBIOS,igual_al_borrador,0.000,ok,True
doc_009,documento,borrador visual LLM (Claude),REVISADO_SIN_CAMBIOS,igual_al_borrador,0.000,ok,True
doc_015,documento,borrador visual LLM (Claude),REVISADO_SIN_CAMBIOS,igual_al_borrador,0.000,ok,True
doc_016,documento,borrador visual LLM (Claude),REVISADO_SIN_CAMBIOS,igual_al_borrador,0.000,ok,True


modalidad,estado,n
audio,REVISADO_CON_CAMBIOS,3
audio,REVISADO_SIN_CAMBIOS,7
documento,REVISADO_SIN_CAMBIOS,10


**Origen de las referencias (declaración de transparencia).** Todas pasaron la validación de consistencia, pero su origen no es homogéneo:
- **10 documentos:** borrador de transcripción visual hecho por un LLM (Claude) a partir de la imagen. La autora los revisó y **aceptó sin cambios** (`REVISADO_SIN_CAMBIOS`).
- **7 audios:** borrador del ASR auxiliar wav2vec2, que no se evalúa. La autora los revisó y **aceptó sin cambios**.
- **3 audios** (aud_002, aud_019 y aud_028): la autora los **corrigió escuchando** (`REVISADO_CON_CAMBIOS`). La columna `magnitud_correccion` mide cuánto difiere la versión final del borrador.

Consecuencias para leer los resultados:
1. El requisito pedía referencias **manuales**; aquí son borradores asistidos y validados por la autora, y así debe declararse.
2. Los errores que hayan quedado en las referencias aceptadas sin cambios —deletreo de wav2vec2, zonas `[inaudible]` o `[ilegible]`— **inflan el CER y el WER de ambos modelos por igual**. Por eso la comparación **relativa** entre modelos es más confiable que el valor absoluto.

## 3. Normalización (idéntica para la referencia y los modelos)
| Aspecto | CER (documentos) | WER (audio) |
|---|---|---|
| Unicode | NFC | NFC |
| Mayúsculas | se convierten a minúsculas | se convierten a minúsculas |
| Puntuación y símbolos | se eliminan (pasan a espacio) | se eliminan (pasan a espacio) |
| Tildes y ñ | **se conservan** (métrica principal); sensibilidad: CER sin tildes, conservando la ñ | se conservan |
| Saltos de línea y espacios | se unifican en un solo espacio | se unifican en un solo espacio |
| Guion + salto de línea | se unen las dos partes de la palabra | — |
| Etiquetas `[...]` | se eliminan (`[ilegible]`) | se eliminan (`[HABLANTE_k]`, `[inaudible]`, `[mm:ss–mm:ss]`) |
| Números | se conservan tal cual | las cifras pasan a palabras (num2words, es) |
| Muletillas | — | se eliminan *eh, em, mm, mmm, ah* |

**Consecuencia importante:** lo que la referencia marca como `[ilegible]` o `[inaudible]` desaparece de ella. Si un modelo produce texto en esa zona, ese texto cuenta como **inserción**. Esto penaliza a ambos modelos por igual en la misma zona.

In [4]:
crono.marca("3. Normalización (autopruebas)")
pruebas = pd.DataFrame(evaluacion.autoverificar(), columns=["Prueba sintética", "OK"])
pruebas["OK"] = pruebas["OK"].map({True: "✅", False: "❌"})
display(pruebas.style.hide(axis="index"))
assert (pruebas["OK"] == "✅").all()
ej = "Lima, 12 de Enero de 1895.\nSe-\nñor D. Pérez [ilegible]"
print("Ejemplo CER  :", repr(evaluacion.normalizar_cer(ej)), "| sin tildes:", repr(evaluacion.normalizar_cer(ej, quitar_tildes=True)))
print("Ejemplo WER  :", repr(evaluacion.normalizar_wer("[00:01.0–00:05.0] [HABLANTE_1] Eh, ganamos 2 a 0, ¿no? [inaudible]")))

Prueba sintética,OK
CER: 1 sustitución en 3 caracteres = 1/3,✅
"CER: puntuación, mayúsculas y saltos de línea no penalizan",✅
CER: guion de corte de línea se une,✅
CER: la tilde cuenta en la métrica principal (1/4),✅
"CER sin tildes: Perú = Peru, pero ñ ≠ n",✅
CER: inserciones pueden superar 1,✅
"WER: 1 sustitución en 4 palabras = 0,25",✅
WER: etiquetas [HABLANTE_k] se eliminan,✅
WER: cifras = palabras,✅
WER: limitación documentada «dos a cero» vs «2-0» = 1 eliminación,✅


Ejemplo CER  : 'lima 12 de enero de 1895 señor d pérez' | sin tildes: 'lima 12 de enero de 1895 señor d perez'
Ejemplo WER  : 'ganamos dos a cero no'


## 4. Cálculo de CER y WER

In [5]:
crono.marca("4. Cálculo de métricas")
R = metricas.calcular()            # guarda metricas_ocr.csv, metricas_asr.csv y resumen_metricas.csv
ocr, asr, resumen, efic = R["ocr"], R["asr"], R["resumen"], R["eficiencia"]
print("Guardado: datos/reportes/metricas_ocr.csv ·", len(ocr), "filas | metricas_asr.csv ·", len(asr),
      "filas | resumen_metricas.csv ·", len(resumen), "filas")

def bootstrap_pareado(df, clave, a, b, n=10_000, semilla=2026):
    # IC 95 % de la diferencia de medias (a - b) remuestreando ELEMENTOS (mismo elemento para ambos modelos)
    piv = df.pivot(index="id", columns="modelo", values=clave)
    d = (piv[a] - piv[b]).to_numpy()
    rng = np.random.default_rng(semilla)
    medias = rng.choice(d, size=(n, len(d)), replace=True).mean(axis=1)
    return d.mean(), np.percentile(medias, 2.5), np.percentile(medias, 97.5), int((d < 0).sum()), int((d > 0).sum())

comparaciones = []   # se guarda en datos/reportes/comparacion_modelos_bootstrap.csv (lo usa el dashboard)
def registrar(modalidad, metrica, a, b, dif, lo, hi, gana_a, gana_b, n):
    comparaciones.append({"modalidad": modalidad, "metrica": metrica, "modelo_a": a, "modelo_b": b,
                          "diferencia_media_a_menos_b": dif, "ic95_inf": lo, "ic95_sup": hi, "significativa": not (lo <= 0 <= hi),
                          "a_mejor": gana_a, "b_mejor": gana_b, "empates": n - gana_a - gana_b, "remuestreos": 10_000, "semilla": 2026})

Guardado: datos/reportes/metricas_ocr.csv · 20 filas | metricas_asr.csv · 20 filas | resumen_metricas.csv · 4 filas


### 4.1 CER por documento (menor = mejor)

In [6]:
t_ocr = ocr.pivot(index=["id", "tipo"], columns="modelo", values="cer").reset_index()
t_ocr["mejor modelo"] = ocr.drop_duplicates("id").set_index("id").loc[t_ocr["id"], "mejor_modelo_del_elemento"].values
t_ocr["[ilegible] en referencia"] = ocr.drop_duplicates("id").set_index("id").loc[t_ocr["id"], "ilegible_en_referencia"].values
t_ocr = t_ocr.sort_values(["tipo", "id"], key=lambda s: s.map({t: i for i, t in enumerate(ORDEN_TIPO)}) if s.name == "tipo" else s)
display(t_ocr.rename(columns={"PaddleOCR": "CER PaddleOCR", "Tesseract": "CER Tesseract"})
        .style.hide(axis="index").format({"CER PaddleOCR": "{:.3f}", "CER Tesseract": "{:.3f}"})
        .bar(subset=["CER PaddleOCR", "CER Tesseract"], vmin=0, vmax=1, color="#cde2fb"))
r_ocr = resumen[resumen["modalidad"] == "documento"].set_index("modelo")
display(r_ocr[["promedio", "ponderado", "promedio_impreso", "promedio_foto", "promedio_manuscrito", "cer_sin_tildes_promedio"]]
        .rename(columns={"promedio": "CER promedio", "ponderado": "CER ponderado por longitud",
                         "promedio_impreso": "CER impresos", "promedio_foto": "CER fotos",
                         "promedio_manuscrito": "CER manuscritos", "cer_sin_tildes_promedio": "CER sin tildes"})
        .style.format("{:.3f}"))
dif, lo, hi, gana_a, gana_b = bootstrap_pareado(ocr, "cer", "PaddleOCR", "Tesseract")
print(f"Diferencia media CER (PaddleOCR − Tesseract) = {dif:+.3f} · IC 95 % bootstrap pareado [{lo:+.3f}, {hi:+.3f}] · "
      f"PaddleOCR mejor en {gana_a}/10, Tesseract mejor en {gana_b}/10")
registrar("documento", "CER", "PaddleOCR", "Tesseract", dif, lo, hi, gana_a, gana_b, ocr["id"].nunique())

id,tipo,CER PaddleOCR,CER Tesseract,mejor modelo,[ilegible] en referencia
doc_005,impreso,0.021,0.000,Tesseract,0
doc_022,impreso,0.005,0.002,Tesseract,0
doc_023,impreso,0.008,0.010,PaddleOCR,0
doc_008,foto,0.722,1.000,PaddleOCR,79
doc_011,foto,0.013,1.000,PaddleOCR,0
doc_019,foto,0.361,0.966,PaddleOCR,8
doc_001,manuscrito,0.742,0.991,PaddleOCR,44
doc_009,manuscrito,0.252,0.911,PaddleOCR,0
doc_015,manuscrito,0.569,0.905,PaddleOCR,4
doc_016,manuscrito,0.258,0.807,PaddleOCR,3


,CER promedio,CER ponderado por longitud,CER impresos,CER fotos,CER manuscritos,CER sin tildes
modelo,,,,,,
PaddleOCR,0.295,0.322,0.011,0.365,0.455,0.292
Tesseract,0.659,0.636,0.004,0.989,0.903,0.659


Diferencia media CER (PaddleOCR − Tesseract) = -0.364 · IC 95 % bootstrap pareado [-0.567, -0.176] · PaddleOCR mejor en 8/10, Tesseract mejor en 2/10


In [7]:
# Gráfico 1 — CER por documento, PaddleOCR vs Tesseract
orden_docs = t_ocr["id"].tolist()
fig = go.Figure()
for modelo, color in COLOR_OCR.items():
    d = ocr[ocr["modelo"] == modelo].set_index("id").loc[orden_docs].reset_index()
    fig.add_trace(go.Bar(x=d["id"] + "<br><span style='font-size:11px'>" + d["tipo"] + "</span>", y=d["cer"], name=modelo,
                         marker=dict(color=color, cornerradius=4),
                         customdata=np.c_[d["S"], d["D"], d["I"], d["N"]],
                         hovertemplate="<b>%{x}</b><br>" + modelo + ": CER %{y:.3f}<br>S %{customdata[0]} · D %{customdata[1]} · "
                                       "I %{customdata[2]} · N %{customdata[3]}<extra></extra>"))
fig.add_hline(y=1, line=dict(color=EJE, width=1))
fig.update_layout(barmode="group", height=470,
    title=dict(text=f"PaddleOCR reduce el CER promedio de {r_ocr.loc['Tesseract','promedio']:.2f} a {r_ocr.loc['PaddleOCR','promedio']:.2f}; "
                    "en impresos ambos son casi perfectos",
               subtitle=dict(text="CER por documento (menor = mejor). La línea en 1,0 corresponde a no reconocer nada; un CER mayor que 1 indica inserciones")),
    yaxis=dict(title="CER", rangemode="tozero"), xaxis=dict(title=None))
fig.show()

In [8]:
# Gráfico 2 — CER promedio por tipo documental
fig = go.Figure()
for modelo, color in COLOR_OCR.items():
    vals = [r_ocr.loc[modelo, f"promedio_{t}"] for t in ORDEN_TIPO]
    fig.add_trace(go.Bar(x=ORDEN_TIPO, y=vals, name=modelo, marker=dict(color=color, cornerradius=4),
                         text=[f"{v:.2f}" for v in vals], textposition="outside", textfont=dict(color=TINTA2),
                         hovertemplate="%{x} · " + modelo + ": CER medio %{y:.3f}<extra></extra>"))
fig.update_layout(barmode="group", height=420,
    title=dict(text="La dificultad crece de impresos a fotos y manuscritos; Tesseract colapsa fuera del impreso",
               subtitle=dict(text="CER promedio por tipo de documento (menor = mejor); 3 impresos, 3 fotos y 4 manuscritos")),
    yaxis=dict(title="CER promedio", range=[0, 1.15]), xaxis=dict(title=None))
fig.show()

**Lectura del CER.**
- **Global:** PaddleOCR obtiene un CER promedio de **0,295** y Tesseract de **0,659**. La diferencia media es de −0,364, con un **IC 95 % por bootstrap pareado de [−0,567, −0,176]**: es **significativa** aun con 10 documentos. PaddleOCR gana en 8 de 10.
- **Impresos:** ambos son casi perfectos (Tesseract 0,004 y PaddleOCR 0,011). En doc_005 y doc_022, **Tesseract es algo mejor**.
- **Fotos y manuscritos:** Tesseract **colapsa**. Devuelve **0 caracteres** en doc_008 (vitrina) y en doc_011 (relieve), por lo que su CER es 1,000 en ambos, y apenas 8 caracteres en doc_001. Es un resultado válido de su configuración estándar sin preprocesamiento, que no se ajustó después de ver los resultados.
- **Tipo de error:** en PaddleOCR el error dominante es la **omisión** (73 % de eliminaciones). En los manuscritos produce aproximadamente la mitad del texto de la referencia.
- **Tildes:** el CER sin tildes casi no cambia (0,292 frente a 0,295), así que los errores no se deben principalmente a las tildes.

### 4.2 WER por clip (menor = mejor)

In [9]:
t_asr = asr.pivot(index=["id", "fuente", "duracion_s"], columns="modelo", values="wer").reset_index()
info = asr.drop_duplicates("id").set_index("id")
t_asr["mejor modelo"] = info.loc[t_asr["id"], "mejor_modelo_del_elemento"].values
t_asr["referencia"] = info.loc[t_asr["id"], "referencia_revisada"].values
t_asr["[inaudible] en ref."] = info.loc[t_asr["id"], "inaudible_en_referencia"].values
display(t_asr.rename(columns={"faster-whisper small": "WER faster-whisper", "Parakeet TDT 0.6B v3": "WER Parakeet",
                              "duracion_s": "duración (s)"})
        .style.hide(axis="index").format({"WER faster-whisper": "{:.3f}", "WER Parakeet": "{:.3f}", "duración (s)": "{:.1f}"})
        .bar(subset=["WER faster-whisper", "WER Parakeet"], vmin=0, vmax=1, color="#cde2fb"))
r_asr = resumen[resumen["modalidad"] == "audio"].set_index("modelo")
display(r_asr[["promedio", "ponderado"]].rename(columns={"promedio": "WER promedio", "ponderado": "WER ponderado"}).style.format("{:.3f}"))
dif, lo, hi, gana_a, gana_b = bootstrap_pareado(asr, "wer", "faster-whisper small", "Parakeet TDT 0.6B v3")
print(f"Diferencia media WER (Whisper − Parakeet) = {dif:+.3f} · IC 95 % bootstrap pareado [{lo:+.3f}, {hi:+.3f}] · "
      f"Whisper mejor en {gana_a}/10, Parakeet mejor en {gana_b}/10, empates {10 - gana_a - gana_b}")
registrar("audio", "WER", "faster-whisper small", "Parakeet TDT 0.6B v3", dif, lo, hi, gana_a, gana_b, asr["id"].nunique())
pd.DataFrame(comparaciones).to_csv(REPORTES / "comparacion_modelos_bootstrap.csv", index=False, encoding="utf-8")
por_rev = asr.groupby(["referencia_revisada", "modelo"])["wer"].mean().unstack()
display(Markdown("**WER medio según el tipo de revisión de la referencia**"))
display(por_rev.style.format("{:.3f}"))

id,fuente,duración (s),WER Parakeet,WER faster-whisper,mejor modelo,referencia,[inaudible] en ref.
aud_002,audio_A,71.3,0.281,0.216,faster-whisper small,REVISADO_CON_CAMBIOS,6
aud_006,audio_A,70.5,0.307,0.347,Parakeet TDT 0.6B v3,REVISADO_SIN_CAMBIOS,14
aud_010,audio_B,65.5,0.338,0.324,faster-whisper small,REVISADO_SIN_CAMBIOS,9
aud_012,audio_C,81.8,0.323,0.332,Parakeet TDT 0.6B v3,REVISADO_SIN_CAMBIOS,14
aud_013,audio_D,82.3,0.343,0.354,Parakeet TDT 0.6B v3,REVISADO_SIN_CAMBIOS,16
aud_019,audio_E,84.5,0.684,0.861,Parakeet TDT 0.6B v3,REVISADO_CON_CAMBIOS,13
aud_022,audio_F,67.8,0.274,0.288,Parakeet TDT 0.6B v3,REVISADO_SIN_CAMBIOS,5
aud_023,audio_F,68.4,0.123,0.123,empate,REVISADO_SIN_CAMBIOS,4
aud_026,audio_F,65.2,0.167,0.167,empate,REVISADO_SIN_CAMBIOS,7
aud_028,audio_G,66.2,0.248,0.248,empate,REVISADO_CON_CAMBIOS,7


,WER promedio,WER ponderado
modelo,,
faster-whisper small,0.326,0.307
Parakeet TDT 0.6B v3,0.309,0.297


Diferencia media WER (Whisper − Parakeet) = +0.017 · IC 95 % bootstrap pareado [-0.014, +0.059] · Whisper mejor en 2/10, Parakeet mejor en 5/10, empates 3


**WER medio según el tipo de revisión de la referencia**

modelo,Parakeet TDT 0.6B v3,faster-whisper small
referencia_revisada,,
REVISADO_CON_CAMBIOS,0.404,0.442
REVISADO_SIN_CAMBIOS,0.268,0.276


In [10]:
# Gráfico 3 — WER por clip, faster-whisper vs Parakeet (gráfico de puntos pareados)
orden = t_asr.sort_values("faster-whisper small")["id"].tolist()
fig = go.Figure()
for _, r in t_asr.iterrows():
    fig.add_trace(go.Scatter(x=[r["faster-whisper small"], r["Parakeet TDT 0.6B v3"]], y=[r["id"]] * 2, mode="lines",
                             line=dict(color=EJE, width=2), hoverinfo="skip", showlegend=False))
for modelo, color in COLOR_ASR.items():
    d = asr[asr["modelo"] == modelo]
    fig.add_trace(go.Scatter(x=d["wer"], y=d["id"], mode="markers", name=modelo,
                             marker=dict(size=12, color=color, line=dict(color=SUPERFICIE, width=2)),
                             customdata=np.c_[d["fuente"], d["S"], d["D"], d["I"], d["N"], d["referencia_revisada"]],
                             hovertemplate="<b>%{y}</b> (%{customdata[0]})<br>" + modelo + ": WER %{x:.3f}<br>S %{customdata[1]} · "
                                           "D %{customdata[2]} · I %{customdata[3]} · N %{customdata[4]}<br>%{customdata[5]}<extra></extra>"))
fig.update_layout(height=470,
    title=dict(text=f"WER parecido entre modelos ({r_asr.loc['Parakeet TDT 0.6B v3','promedio']:.2f} Parakeet vs "
                    f"{r_asr.loc['faster-whisper small','promedio']:.2f} Whisper); aud_019, con música, es el caso extremo",
               subtitle=dict(text="WER por clip (menor = mejor); cada línea une los dos modelos sobre el mismo clip")),
    xaxis=dict(title="WER", rangemode="tozero"), yaxis=dict(categoryorder="array", categoryarray=orden, title=None))
fig.show()

**Lectura del WER.**
- **Global:** Parakeet obtiene un WER promedio de **0,309** y faster-whisper de **0,326**. La diferencia media es de +0,017, con un **IC 95 % de [−0,014, +0,059]**: **no es significativa**, porque el intervalo incluye el 0. Parakeet gana en 5 clips, Whisper en 2 y empatan en 3.
- **El nivel absoluto, alrededor de 0,3, está inflado por las referencias.** Entre el 42 y el 44 % de los errores son **inserciones**, y cada referencia tiene entre 4 y 16 `[inaudible]`. Lo que los modelos transcriben en esas zonas cuenta como error. Además, los dos modelos difieren entre sí mucho menos (desacuerdo de 0,05–0,16) de lo que difieren de la referencia, lo que apunta a errores en la propia referencia.
- **aud_019 (ceremonia con música) muestra dos modos de fallo opuestos:**
  - **Whisper sustituye e inserta** (S = 39, I = 28): genera texto que no está en la referencia, lo que es compatible con **alucinar**.
  - **Parakeet omite** (D = 44): solo 35 palabras frente a 79 de la referencia.

## 5. Eficiencia: tiempo, RTF, latencia y estabilidad
Los cuatro modelos se midieron en **la misma sesión** (`scripts/11_benchmark_tiempos.py`), en CPU (Intel Core Ultra 5 125H) y sin otros procesos pesados: una pasada por elemento después de un calentamiento. **Estabilidad** significa que repetir la ejecución produce exactamente el mismo texto.

In [11]:
crono.marca("5. Eficiencia")
trans = pd.read_csv(DATOS / "transcripciones.csv").set_index("id")
ids_audio = muestra.loc[muestra["modalidad"] == "audio", "id"]
lat_w = trans.loc[ids_audio, "latencia_primer_segmento_s"].median()
def tam_mb(p):
    p = Path(p); return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6 if p.exists() else float("nan")
tam = {"PaddleOCR": tam_mb(RAIZ / "modelos/paddlex/official_models"), "Tesseract": tam_mb(RAIZ / "modelos/tessdata"),
       "faster-whisper small": tam_mb(RAIZ / "modelos/faster-whisper"),
       "Parakeet TDT 0.6B v3": tam_mb(RAIZ / "modelos/asr_parakeet")}
ef = efic.set_index("modelo")
tabla_ef = pd.DataFrame({
    "tiempo mediano por elemento (s)": ef["tiempo_mediano_s"], "tiempo total muestra (s)": ef["tiempo_total_s"],
    "RTF mediano": ef["rtf_mediano"], "RTF máximo": ef["rtf_max"], "carga del modelo (s)": ef["carga_modelo_s"],
    "estabilidad": ef["estabilidad"], "tamaño en disco (MB)": pd.Series(tam)})
tabla_ef.loc["faster-whisper small", "latencia (s)"] = lat_w
tabla_ef.loc["Parakeet TDT 0.6B v3", "latencia (s)"] = ef.loc["Parakeet TDT 0.6B v3", "tiempo_mediano_s"]
display(tabla_ef.style.format("{:.2f}", na_rep="—", subset=[c for c in tabla_ef.columns if c != "estabilidad"]))
print("Latencia: Whisper = mediana hasta el primer segmento (Parte 1, procesa en flujo); "
      "Parakeet = modelo offline, entrega el texto al terminar el clip (latencia = tiempo total).")

,tiempo mediano por elemento (s),tiempo total muestra (s),RTF mediano,RTF máximo,carga del modelo (s),estabilidad,tamaño en disco (MB),latencia (s)
PaddleOCR,109.69,1094.21,—,—,2.97,10/10,139.16,—
Parakeet TDT 0.6B v3,10.57,92.04,0.14,0.16,1.88,10/10,671.24,10.57
Tesseract,3.99,33.25,—,—,—,no medida,24.13,—
faster-whisper small,19.88,230.91,0.29,0.41,1.70,9/10,486.21,4.89


Latencia: Whisper = mediana hasta el primer segmento (Parte 1, procesa en flujo); Parakeet = modelo offline, entrega el texto al terminar el clip (latencia = tiempo total).


In [12]:
# Gráfico 4 — Tiempo por elemento (OCR) y RTF por clip (ASR): dos medidas distintas, dos paneles
bench = pd.read_csv(REPORTES / "benchmark_tiempos.csv"); bench = bench[bench["id"] != "_carga_modelo"]
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.14,
                    subplot_titles=("OCR · segundos por documento (escala log)", "ASR · RTF por clip"))
for modelo, color in COLOR_OCR.items():
    d = bench[bench["modelo"] == modelo]
    fig.add_trace(go.Box(x=d["tiempo_s"], y=[modelo] * len(d), orientation="h", boxpoints="all", jitter=0.3, pointpos=0,
                         marker=dict(color=color, size=9, line=dict(color=SUPERFICIE, width=1.5)), line=dict(color=color, width=1.5),
                         fillcolor="rgba(0,0,0,0)", name=modelo, showlegend=False, customdata=d["id"],
                         hovertemplate="%{customdata}: %{x:.1f} s<extra>" + modelo + "</extra>"), 1, 1)
for modelo, color in COLOR_ASR.items():
    d = bench[bench["modelo"] == modelo]
    fig.add_trace(go.Box(x=d["rtf"], y=[modelo.split(" ")[0]] * len(d), orientation="h", boxpoints="all", jitter=0.3, pointpos=0,
                         marker=dict(color=color, size=9, line=dict(color=SUPERFICIE, width=1.5)), line=dict(color=color, width=1.5),
                         fillcolor="rgba(0,0,0,0)", name=modelo, showlegend=False, customdata=d["id"],
                         hovertemplate="%{customdata}: RTF %{x:.3f}<extra>" + modelo + "</extra>"), 1, 2)
fig.update_xaxes(type="log", title_text="segundos (menor = mejor)", tickvals=[0.5, 1, 2, 5, 10, 20, 50, 100, 200], row=1, col=1)
fig.update_xaxes(title_text="RTF (menor = mejor; < 1 = más rápido que tiempo real)", rangemode="tozero", row=1, col=2)
fig.update_layout(height=400,
    title=dict(text=f"Tesseract es ~{ef.loc['PaddleOCR','tiempo_mediano_s']/ef.loc['Tesseract','tiempo_mediano_s']:.0f}× más rápido que PaddleOCR; "
                    f"Parakeet tiene la mitad del RTF de Whisper",
               subtitle=dict(text="Medición controlada: los cuatro modelos en la misma sesión de CPU, una pasada por elemento después de un calentamiento")))
fig.update_annotations(font=dict(size=13, color=TINTA2))
fig.show()

**Eficiencia.**
- **OCR:** Tesseract necesita **≈ 4 s** por documento y PaddleOCR **≈ 110 s** (mediana), unas **27 veces más**. En la Parte 1 PaddleOCR llegó a 9,5 h en el mapa doc_025, por ejecutarse en CPU sin oneDNN.
- **ASR:** el RTF mediano de Parakeet (**0,14**) es la mitad del de Whisper (**0,29**). Ambos procesan mucho más rápido que el tiempo real en CPU.
- **Estabilidad:** PaddleOCR y Parakeet repitieron exactamente su salida (10/10). Whisper cambió en aud_019 (9/10), coherente con la decodificación con temperatura que aplica en audio difícil.
- **Latencia:** Whisper entrega el primer segmento en unos pocos segundos porque procesa en flujo. Parakeet es un modelo offline y entrega el texto al terminar el clip.

## 6. Casos difíciles y análisis de errores

In [13]:
crono.marca("6. Casos difíciles")
dif_docs = ocr.groupby(["id", "tipo"])["cer"].min().sort_values(ascending=False).head(3).rename("CER del mejor modelo").reset_index()
dif_aud = asr.groupby(["id", "fuente"])["wer"].min().sort_values(ascending=False).head(3).rename("WER del mejor modelo").reset_index()
display(Markdown("**3 documentos más difíciles** (aun el mejor modelo falla)"))
display(dif_docs.style.hide(axis="index").format({"CER del mejor modelo": "{:.3f}"}))
display(Markdown("**3 audios más difíciles**"))
display(dif_aud.style.hide(axis="index").format({"WER del mejor modelo": "{:.3f}"}))
desc = (pd.concat([ocr.assign(m="OCR"), asr.assign(m="ASR")])
        .groupby(["m", "modelo"])[["S", "D", "I", "N"]].sum())
desc["% sustituciones"] = 100 * desc["S"] / desc[["S", "D", "I"]].sum(axis=1)
desc["% eliminaciones"] = 100 * desc["D"] / desc[["S", "D", "I"]].sum(axis=1)
desc["% inserciones"] = 100 * desc["I"] / desc[["S", "D", "I"]].sum(axis=1)
display(Markdown("**Tipo de error (sobre toda la muestra)**: las eliminaciones indican texto no reconocido y las inserciones, texto de más"))
display(desc[["% sustituciones", "% eliminaciones", "% inserciones"]].style.format("{:.0f} %"))

**3 documentos más difíciles** (aun el mejor modelo falla)

id,tipo,CER del mejor modelo
doc_001,manuscrito,0.742
doc_008,foto,0.722
doc_015,manuscrito,0.569


**3 audios más difíciles**

id,fuente,WER del mejor modelo
aud_019,audio_E,0.684
aud_013,audio_D,0.343
aud_010,audio_B,0.324


**Tipo de error (sobre toda la muestra)**: las eliminaciones indican texto no reconocido y las inserciones, texto de más

In [14]:
# Ejemplos de alineación (solo documentos históricos de dominio público: no contienen datos personales protegidos)
def ejemplo(id_doc, modelo, n=320):
    ref = (RAW / "referencias" / "documentos" / f"{id_doc}.txt").read_text(encoding="utf-8")
    hip = json.loads((RAW / "cache" / metricas.OCR[modelo].name / f"{id_doc}.json").read_text(encoding="utf-8"))["texto_ocr"]
    r, h = evaluacion.normalizar_cer(ref)[:n], evaluacion.normalizar_cer(hip)[:n]
    display(Markdown(f"**{id_doc} · {modelo}** (primeros {n} caracteres normalizados)\n\n```text\nREF: {r}\nOCR: {h}\n```"))
ejemplo("doc_015", "PaddleOCR"); ejemplo("doc_015", "Tesseract"); ejemplo("doc_011", "PaddleOCR")

**doc_015 · PaddleOCR** (primeros 320 caracteres normalizados)

```text
REF: exmo s r d agustin gamarra 12 de agosto de 1840 mi respetable gral y padrino sin embargo de hallarme bien enfermo con la disenteria q he contraido en mi confinacion de abancay me he situado en este pueblo en de la orden suprema si mi mal como es probable se agraba me sera permitido bajar á urubamba afin de reparar mi s
OCR: suniti 12 me agosk vu1840 mi reppetabe hal ypadrino la diventiria a he lontraido en mi confi nacion de tbaneay me he lituado in este ba me sena pernitid eajar a bubambay no me canvare in zecomemdan a to in honrsades ul comandante tiabima aste jege eh katallon lmeo t basin al enta indicacion ceo eumplin con lo pusticin 
```

**doc_015 · Tesseract** (primeros 320 caracteres normalizados)

```text
REF: exmo s r d agustin gamarra 12 de agosto de 1840 mi respetable gral y padrino sin embargo de hallarme bien enfermo con la disenteria q he contraido en mi confinacion de abancay me he situado en este pueblo en de la orden suprema si mi mal como es probable se agraba me sera permitido bajar á urubamba afin de reparar mi s
OCR: ea a al e jo laa mb cae la reia das 7 la e ac an as ei o ui di xme a ae e
```

**doc_011 · PaddleOCR** (primeros 320 caracteres normalizados)

```text
REF: carta escrita por el general felipe santiago salaverry horas antes de su muerte arequipa febrero 17 1886 mi querida esposa dentro de pocos momentos voy a ser pasado por las armas y te debo el ultimo adios es este tu conocias bien mi corazon y no puedes dudar de que mis intenciones en toda mi vida publica han sido muy p
OCR: carta escrita porelgeneral felipesantiagosalaverry horasantesdesumuerte arequipa febrero17 1886 mi querida esposa dentro de pocos momentos voy a ser pagado por las armas y te debo el ultimo adios es este tu conocias bien mi corazon y no puedes dudar de que mis intengiones en toda mi vida publica han sido muy puras ella
```

**Casos difíciles y causas probables.**
- **doc_001 (manuscrito colonial de 1821, CER 0,742):** la imagen es de baja resolución, tiene firmas superpuestas y una caligrafía antigua. La referencia tiene 44 `[ilegible]`, porque ni la lectura humana asistida lo resolvió.
- **doc_008 (tabla de 1879 bajo vidrio, 0,722):** combina reflejos, letra manuscrita diminuta y 79 `[ilegible]`. Además, en tablas el CER **penaliza el orden de lectura**: la referencia recorre la tabla por secciones y PaddleOCR por filas, de modo que penaliza incluso palabras bien reconocidas.
- **doc_015 (carta manuscrita de 1840, 0,569):** cursiva con abreviaturas. PaddleOCR omite casi la mitad del texto.
- **aud_019 (ceremonia con música, 0,684 con el mejor modelo):** la música de fondo y varias voces provocan los dos modos de fallo descritos arriba.
- **aud_013 y aud_010 (alrededor de 0,33):** entrevistas en la calle con ruido. Sus referencias son borradores aceptados sin cambios con 9–16 `[inaudible]`, así que parte del error se debe a la referencia.
- **aud_028 (multihablante, 0,248 en ambos modelos):** los cambios de voz no degradan de forma notable el WER. El problema aquí está en la diarización, que estimó 2 hablantes cuando hay al menos 3, como se vio en la Parte 1.
- **Fotos con buena tipografía:** la placa en relieve doc_011 da un CER de 0,013 con PaddleOCR. La dificultad de una foto depende más de los reflejos y la escritura que de ser una foto.

## 7. Comparación final y recomendación
Las tablas combinan precisión (CER/WER, menor es mejor), velocidad, estabilidad y costo. Los textos de ventajas y desventajas se apoyan en los números medidos arriba.

In [15]:
crono.marca("7. Comparación final")
comp_ocr = pd.DataFrame({
    "Modelo": ["PaddleOCR", "Tesseract"],
    "CER promedio": [r_ocr.loc[m, "promedio"] for m in ["PaddleOCR", "Tesseract"]],
    "CER impresos": [r_ocr.loc[m, "promedio_impreso"] for m in ["PaddleOCR", "Tesseract"]],
    "CER fotos": [r_ocr.loc[m, "promedio_foto"] for m in ["PaddleOCR", "Tesseract"]],
    "CER manuscritos": [r_ocr.loc[m, "promedio_manuscrito"] for m in ["PaddleOCR", "Tesseract"]],
    "Tiempo mediano (s/doc)": [ef.loc[m, "tiempo_mediano_s"] for m in ["PaddleOCR", "Tesseract"]],
    "Costo computacional": [f"alto: CPU sin oneDNN, {tam['PaddleOCR']:.0f} MB de modelos, tiempo muy variable (hasta 9,5 h en un mapa)",
                            f"bajo: {tam['Tesseract']:.0f} MB, segundos por página"],
    "Ventajas": ["robusto en fotos y manuscritos; detección de líneas en escenas complejas; salida estable 10/10",
                 "muy rápido y liviano; excelente en impresos limpios"],
    "Desventajas": ["lento y con tiempos impredecibles en CPU; une palabras sin espacio en algunos relieves",
                    "no detecta texto en fotos con vidrio o relieve (0 palabras en 2 de 3 fotos); casi inútil en manuscritos"]})
comp_asr = pd.DataFrame({
    "Modelo": ["faster-whisper small", "Parakeet TDT 0.6B v3"],
    "WER promedio": [r_asr.loc[m, "promedio"] for m in ["faster-whisper small", "Parakeet TDT 0.6B v3"]],
    "Latencia (s)": [lat_w, ef.loc["Parakeet TDT 0.6B v3", "tiempo_mediano_s"]],
    "RTF mediano": [ef.loc[m, "rtf_mediano"] for m in ["faster-whisper small", "Parakeet TDT 0.6B v3"]],
    "Estabilidad": [ef.loc[m, "estabilidad"] for m in ["faster-whisper small", "Parakeet TDT 0.6B v3"]],
    "Costo computacional": [f"medio: {tam['faster-whisper small']:.0f} MB (int8), CPU", f"medio: {tam['Parakeet TDT 0.6B v3']:.0f} MB (int8 ONNX), CPU"],
    "Ventajas": ["detección y forzado de idioma, VAD, marcas por palabra (diarización), salida en flujo con baja latencia",
                 "≈2× más rápido; salida idéntica al repetir; sin dependencia de torch"],
    "Desventajas": ["más lento; no determinista en audio difícil (aud_019 cambió); riesgo de alucinar en música",
                    "no permite forzar el idioma; puede omitir habla en audio con música (35 palabras en aud_019)"]})
display(comp_ocr.style.hide(axis="index").format({c: "{:.3f}" for c in ["CER promedio", "CER impresos", "CER fotos", "CER manuscritos"]}
                                                 | {"Tiempo mediano (s/doc)": "{:.1f}"}))
display(comp_asr.style.hide(axis="index").format({"WER promedio": "{:.3f}", "Latencia (s)": "{:.1f}", "RTF mediano": "{:.3f}"}))

Modelo,CER promedio,CER impresos,CER fotos,CER manuscritos,Tiempo mediano (s/doc),Costo computacional,Ventajas,Desventajas
PaddleOCR,0.295,0.011,0.365,0.455,109.7,"alto: CPU sin oneDNN, 139 MB de modelos, tiempo muy variable (hasta 9,5 h en un mapa)",robusto en fotos y manuscritos; detección de líneas en escenas complejas; salida estable 10/10,lento y con tiempos impredecibles en CPU; une palabras sin espacio en algunos relieves
Tesseract,0.659,0.004,0.989,0.903,4.0,"bajo: 24 MB, segundos por página",muy rápido y liviano; excelente en impresos limpios,no detecta texto en fotos con vidrio o relieve (0 palabras en 2 de 3 fotos); casi inútil en manuscritos


Modelo,WER promedio,Latencia (s),RTF mediano,Estabilidad,Costo computacional,Ventajas,Desventajas
faster-whisper small,0.326,4.9,0.286,9/10,"medio: 486 MB (int8), CPU","detección y forzado de idioma, VAD, marcas por palabra (diarización), salida en flujo con baja latencia",más lento; no determinista en audio difícil (aud_019 cambió); riesgo de alucinar en música
Parakeet TDT 0.6B v3,0.309,10.6,0.143,10/10,"medio: 671 MB (int8 ONNX), CPU",≈2× más rápido; salida idéntica al repetir; sin dependencia de torch,no permite forzar el idioma; puede omitir habla en audio con música (35 palabras en aud_019)


**Recomendación.**
- **OCR, estrategia híbrida:** **Tesseract para impresos limpios** (igual de preciso, unas 27 veces más rápido y liviano) y **PaddleOCR para fotos y manuscritos**, donde es el único con resultados útiles (CER 0,37 y 0,46 frente a 0,99 y 0,90).
- **Manuscritos, problema abierto:** con un CER cercano a 0,46, ningún OCR los resuelve. Haría falta un modelo de reconocimiento de escritura manuscrita (HTR); esto queda como trabajo futuro y no se evaluó aquí.
- **ASR:** la precisión es estadísticamente equivalente, así que el criterio decisivo es práctico. Se recomienda **Parakeet para transcribir lotes**, porque es dos veces más rápido y totalmente repetible. **Whisper** sigue siendo preferible cuando se necesita forzar el idioma, baja latencia en flujo o marcas por palabra para la diarización. En audio con música, ninguno es confiable sin revisión humana.

## 8. Extracción estructurada con LLM y auditoría humana
- **Modelo:** `qwen2.5:3b` (Ollama, local y gratuito), con `temperature=0`, `seed=42` y salida forzada a un esquema JSON.
- **Entrada:** el texto OCR o ASR **seudonimizado** de los 55 elementos del corpus, de modo que el LLM no ve datos personales.
- **Campos:** `modalidad`, `tema` y `resumen`. En documentos, además, `emisor`, `fecha` y `monto`, solo si aparecen. En audios, `citas_clave` y `numero_hablantes`. Si un dato no existe, `null`.
- **Verificaciones automáticas contra invenciones:** que las citas sean literales, que las cifras de fecha y monto estén en el texto y que el número de hablantes coincida con las etiquetas de diarización.

In [16]:
crono.marca("8. LLM y auditoría")
ext = pd.read_csv(DATOS / "extraccion_llm.csv")
aud_llm = pd.read_csv(DATOS / "auditoria_llm.csv")
a = ext[ext["modalidad"] == "audio"]
lit = a["verif_citas"].str.extract(r"(\d+)/(\d+)").astype(int)
resumen_llm = pd.DataFrame([
    ("Extracciones", f"{len(ext)} ({(ext['modalidad']=='documento').sum()} documentos + {len(a)} audios)"),
    ("JSON válido y con todas las claves", f"{ext['json_valido'].sum()}/{len(ext)}"),
    ("Errores de parsing", f"{(ext['error_parsing'].fillna('') != '').sum()}"),
    ("Citas clave literales en el texto fuente", f"{lit[0].sum()}/{lit[1].sum()} ({100*lit[0].sum()/lit[1].sum():.0f} %)"),
    ("numero_hablantes = etiquetas de diarización", f"{(a['verif_hablantes']=='ok').sum()}/{len(a)}"),
    ("Documentos con cifras de fecha/monto no halladas", f"{(ext.loc[ext['modalidad']=='documento','verif_fecha_monto']!='ok').sum()}/{(ext['modalidad']=='documento').sum()}"),
    ("Tiempo mediano por extracción", f"{ext['tiempo_s'].median():.1f} s"),
    ("Modelo", ext["modelo_llm"].iloc[0]),
], columns=["Indicador", "Valor"])
display(resumen_llm.style.hide(axis="index"))
campos = ext[ext["modalidad"] == "documento"].assign(c=lambda d: d["campos_json"].map(json.loads))
for k in ["emisor", "fecha", "monto"]:
    campos[f"tiene_{k}"] = campos["c"].map(lambda c: c.get(k) is not None)
print("Documentos con emisor/fecha/monto no nulos:", {k: int(campos[f"tiene_{k}"].sum()) for k in ["emisor", "fecha", "monto"]})
for i in ["doc_011", "doc_014", "doc_016"]:
    display(Markdown(f"**{i}** → `{ext.set_index('id').loc[i, 'campos_json']}`"))

Indicador,Valor
Extracciones,55 (25 documentos + 30 audios)
JSON válido y con todas las claves,55/55
Errores de parsing,0
Citas clave literales en el texto fuente,84/89 (94 %)
numero_hablantes = etiquetas de diarización,14/30
Documentos con cifras de fecha/monto no halladas,0/25
Tiempo mediano por extracción,5.4 s
Modelo,"qwen2.5:3b (Ollama 0.40.8; 3.1B, Q4_K_M; temperature=0, seed=42)"


Documentos con emisor/fecha/monto no nulos: {'emisor': 12, 'fecha': 19, 'monto': 2}


**doc_011** → `{"modalidad": "documento", "tema": "testamento y despedida", "resumen": "El emisor, Felipe Santiago Salaverry, escribe una carta de despedida a su esposa, Juanita, y le deja su testamento.", "emisor": "FELIPI SANTIAGO SALAVERRY", "fecha": "AREQUIPA,FEBRERO17-1886", "monto": null}`

**doc_014** → `{"modalidad": "documento", "tema": "movimientos bancarios", "resumen": "Se trata de un documento que considera las ventajas de un movimiento bancario y menciona un incidente importante.", "emisor": "Vn. D Joming Frictan", "fecha": "Vrallyua Cnuarze 2de1822", "monto": null}`

**doc_016** → `{"modalidad": "documento", "tema": "recomendación de funcionario", "resumen": "El documento recomienda a un funcionario para un cargo", "emisor": null, "fecha": null, "monto": null}`

### 8.1 Auditoría humana de 20 extracciones
La autora revisó 10 documentos y 10 audios, elegidos con la semilla 20261025, contra el texto fuente que recibió el LLM, y respondió **SI** (extracción fiel) o **NO** con una observación (`datos/respuestas_auditoria_llm.txt`). El **acuerdo LLM-humano** es la proporción de extracciones juzgadas correctas. Se acompaña de un **IC 95 % de Wilson**, adecuado para n pequeño. La guía con los criterios y las alertas automáticas está en `datos/GUIA_AUDITORIA_LLM.md`.

In [17]:
# Acuerdo recalculado en vivo desde auditoria_llm.csv y comparado con el reporte versionado
spec16 = importlib.util.spec_from_file_location("m16", RAIZ / "scripts" / "16_guia_auditoria_llm.py")
m16 = importlib.util.module_from_spec(spec16); spec16.loader.exec_module(m16)
decididos = aud_llm["correcto"].isin(["SI", "NO"])
print(f"Auditoría: {len(aud_llm)} casos · decididos por la autora: {decididos.sum()} · pendientes: {(~decididos).sum()}")
if decididos.all():
    acuerdo = m16.acuerdo_por_modalidad(aud_llm)
    reporte = pd.read_csv(REPORTES / "acuerdo_llm_humano.csv")
    print("¿Coincide con datos/reportes/acuerdo_llm_humano.csv?:", np.allclose(acuerdo["acuerdo"], reporte["acuerdo"]))
    display(acuerdo.assign(ic95=lambda d: [f"[{a:.2f}, {b:.2f}]" for a, b in zip(d["ic95_inf"], d["ic95_sup"])])
            .drop(columns=["ic95_inf", "ic95_sup"])
            .rename(columns={"modalidad": "Modalidad", "n": "Casos", "correctos": "SI", "incorrectos": "NO",
                             "acuerdo": "Acuerdo LLM-humano", "ic95": "IC 95 % (Wilson)"})
            .style.hide(axis="index").format({"Acuerdo LLM-humano": "{:.0%}"}))
else:
    print("Acuerdo LLM-humano: Pendiente de revisión (no se calcula hasta que la autora complete la columna 'correcto').")

Auditoría: 20 casos · decididos por la autora: 20 · pendientes: 0
¿Coincide con datos/reportes/acuerdo_llm_humano.csv?: True


Modalidad,Casos,SI,NO,Acuerdo LLM-humano,IC 95 % (Wilson)
documento,10,4,6,40%,"[0.17, 0.69]"
audio,10,6,4,60%,"[0.31, 0.83]"
total,20,10,10,50%,"[0.30, 0.70]"


In [18]:
# ¿Las alertas automáticas anticipan la decisión humana? (la autora veía las alertas: no es una prueba independiente)
docs_meta = pd.read_csv(DATOS / "documentos.csv").set_index("id")
ext_i = ext.set_index("id")
def n_alertas(r):
    es_doc = r["modalidad"] == "documento"
    al = m16.alertas(ext_i.loc[r["id"]], json.loads(r["campos_json"]), str(r["texto_fuente"]),
                     float(docs_meta.loc[r["id"], "confianza_media"]) if es_doc else None,
                     docs_meta.loc[r["id"], "tipo"] if es_doc else "audio")
    return 0 if al[0].startswith("sin alertas") else len(al)
aud_llm["alertas_automaticas"] = aud_llm.apply(n_alertas, axis=1)
cruce = pd.crosstab(np.where(aud_llm["alertas_automaticas"] > 0, "con alertas", "sin alertas"),
                    aud_llm["correcto"]).rename_axis(index="Alertas", columns="Decisión de la autora")
display(cruce)
print("Sin alertas pero juzgados NO:", aud_llm.loc[(aud_llm["alertas_automaticas"] == 0) & (aud_llm["correcto"] == "NO"), "id"].tolist())
display(aud_llm[["id", "modalidad", "alertas_automaticas", "correcto", "observacion"]].fillna("")
        .style.hide(axis="index").set_properties(subset=["observacion"], **{"text-align": "left"}))

Decisión de la autora,NO,SI
Alertas,,
con alertas,8,0
sin alertas,2,10


Sin alertas pero juzgados NO: ['doc_013', 'doc_019']


id,modalidad,alertas_automaticas,correcto,observacion
doc_003,documento,1,NO,revisar monto 487/45; no está claro que sea un monto monetario
doc_007,documento,0,SI,
doc_008,documento,2,NO,"la fecha aparece como ""n/a"" y debería ser null"
doc_012,documento,0,SI,
doc_013,documento,0,NO,"el tema ""promoción de productos y servicios"" no parece estar claramente sustentado por el texto fuente"
doc_015,documento,1,NO,el resumen interpreta demasiado un OCR muy deteriorado
doc_016,documento,0,SI,
doc_017,documento,0,SI,
doc_019,documento,0,NO,el tema y el resumen parecen mezclar fragmentos distintos del panel
doc_025,documento,1,NO,el texto está truncado y la extracción no es suficientemente confiable


**Extracción estructurada.**
- **Formato:** el LLM local produjo **JSON válido en 55 de 55 casos** gracias a la salida forzada por esquema.
- **Citas:** el **94 %** son literales; las 5 restantes están marcadas como posible paráfrasis o invención.
- **Número de hablantes:** solo coincide con la diarización en **14 de 30** audios; el modelo tiende a sumar uno de más.
- **Documentos:** la calidad depende de la del OCR. Sobre un OCR ilegible, como el del manuscrito doc_014, el LLM **inventa un tema** ("movimientos bancarios") y toma texto basura como emisor.

**Acuerdo LLM-humano (auditoría de la autora, 20 casos).**
- **Total: 10/20 = 50 %** (IC 95 % de Wilson [0,30, 0,70]). **Documentos: 4/10 = 40 %** [0,17, 0,69]. **Audios: 6/10 = 60 %** [0,31, 0,83]. Con 10 casos por modalidad los intervalos se solapan, así que la diferencia entre modalidades es **indicativa, no concluyente**.
- **Motivos de los 6 NO en documentos, según las observaciones de la autora:**
  - interpretación excesiva de un OCR deteriorado: el tema de doc_013 y el resumen de doc_015;
  - mezcla de fragmentos distintos de un panel (doc_019);
  - texto truncado a 8000 caracteres (doc_025);
  - un "monto" que no es claramente dinero (doc_003);
  - ausencia escrita como `"n/a"` en vez de `null` (doc_008).
- **Motivos de los 4 NO en audios:**
  - `numero_hablantes` distinto de las etiquetas de diarización (aud_001, aud_017 y aud_027); en los tres casos el LLM **suma uno de más**;
  - citas no literales (aud_006 y aud_027).
- **Las alertas automáticas funcionan como filtro:** los 8 casos con alertas fueron juzgados NO, y de los 12 sin alertas, 10 fueron SI. Los 2 NO sin alertas (doc_013 y doc_019) son errores **semánticos** —tema mal sustentado o fragmentos mezclados— que las reglas no pueden detectar. Matiz: la autora veía las alertas al decidir, así que esta concordancia no es una validación independiente.

## 9. Conclusiones, limitaciones y pendientes

**Conclusiones respecto de la pregunta guía.**
1. **Sí, para impresos y audio de habla clara:** el texto obtenido es buscable y casi exacto (CER ≈ 0,01 en impresos y WER ≈ 0,12–0,17 en los clips más limpios).
2. **Parcialmente, para fotos y manuscritos:** PaddleOCR recupera texto útil en fotos con buena tipografía, pero en manuscritos omite cerca de la mitad del contenido. Allí un archivo "buscable" exige revisión humana o un modelo de escritura manuscrita.
3. **Para audio con música o varias voces** ningún ASR es confiable sin revisión, y la diarización automática subestima el número de hablantes.
4. **La estructuración con un LLM pequeño** funciona en formato (55/55 JSON válidos), pero no en fidelidad. La autora juzgó correctas solo **10 de 20** extracciones (**40 %** en documentos y **60 %** en audios). El modelo hereda los errores del OCR, interpreta de más los textos deteriorados y cuenta mal los hablantes. Por eso los campos estructurados sirven para **orientar la búsqueda**, no como dato verificado. Las alertas automáticas permiten priorizar la revisión humana, pero no reemplazarla.

**Limitaciones técnicas.**
1. **Referencias:** son borradores asistidos (LLM para documentos y wav2vec2 para audio) aceptados en su mayoría sin cambios. Las métricas absolutas deben leerse con cautela; las comparaciones relativas entre modelos son más robustas.
2. **Tamaño y entorno:** con solo 10 elementos por modalidad los intervalos son amplios, y la diferencia entre ASR no es significativa. Además, PaddleOCR corrió en CPU sin oneDNN, por un error de compatibilidad, con tiempos muy variables.
3. **Auditoría del LLM:** la hizo una sola revisora, la autora, que veía las alertas automáticas, así que no hay acuerdo entre anotadores. Además, 20 casos dan intervalos amplios: el acuerdo real podría estar entre el 30 % y el 70 %.

**Estado de la Parte 2:** todas las métricas están calculadas con datos reales, y la auditoría humana está completa (20/20 decididos). Lo único que sigue pendiente de revisión humana es heredado de la Parte 1: la fuente de 6 documentos (doc_005, 006, 007, 009, 010 y 021).

In [19]:
crono.marca(None)
tiempos = pd.DataFrame(crono.registros)
tiempos.to_csv(REPORTES / "tiempos_notebook_02.csv", index=False, encoding="utf-8")
display(tiempos.style.hide(axis="index").format({"segundos": "{:.1f}"}))
print(f"Tiempo total de este notebook: {tiempos['segundos'].sum():.1f} s")

seccion,segundos
0. Configuración,0.4
1. Muestra congelada,0.5
2. Referencias y trazabilidad,0.1
3. Normalización (autopruebas),0.0
4. Cálculo de métricas,1.4
5. Eficiencia,0.2
6. Casos difíciles,0.0
7. Comparación final,0.0
8. LLM y auditoría,0.1


Tiempo total de este notebook: 2.7 s
